# Resource-Aware Deep Learning: Hands-On

This notebook will teach you, how to track your own (deep-learning) pipeline's resource consumption using tools like [↗CodeCarbon](https://codecarbon.io/), [↗MLFlow](https://mlflow.org/), and the [↗Lamarr Energy Tracker (LET)](https://github.com/lamarr-institute/lamarr-energy-tracker). This tutorial is built on [↗PyTorch](https://pytorch.org) and [↗PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/). Please make sure you have installed and activated the provided virtual environment (either through `(uv) venv` or `conda/mamba`).

We have provided a small Python package `resource_aware_ml` that implements a number of SRResNet architectures inside the `resource_aware_ml.architectures.model` submodule. If you have not yet installed the package, please do so now inside the repository root directory, e.g. using [uv](https://docs.astral.sh/uv/):

```shell-session
$ uv pip install -e .
```

We can now load the `models` submodule, which implements the following architectures in decreasing complexity:
```
SRResNet18
SRResNet10
SRResNet6
SRResNet4
```
![SRResNet Overview](assets/resnet_sc.png)

In [1]:
from rich import print  # nicer prints

from resource_aware_ml.architectures import models

Choose one of the architectures. Beware that higher complexity will mean more GPU memory consumption. Make sure, you initialise the model; the print function below should print an overview of the architecture.

In [2]:
# Choose your architecture here
model = models.SRResNet6()

print(model)

SRResNet6(
  (preBlock): Sequential(
    (0): Conv2d(2, 64, kernel_size=(9, 9), stride=(1, 1), padding=(4, 4), groups=2)
    (1): PReLU(num_parameters=1)
  )
  (final): Sequential(
    (0): Conv2d(64, 2, kernel_size=(9, 9), stride=(1, 1), padding=(4, 4), groups=2)
    (1): PReLU(num_parameters=1)
  )
  (blocks): Sequential(
    (0): SRBlock(
      (idconv): Identity()
      (pool): Identity()
      (convs): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
        (1): Dropout(p=False, inplace=False)
        (2): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (3): PReLU(num_parameters=1)
        (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
        (5): Dropout(p=False, inplace=False)
        (6): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      )
    )
    (1): SRBlock(
      (idconv): Identity()
      (pool): Identity()
      (convs): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
        (1): Dropout(p=False, inplace=False)
        (2): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (3): PReLU(num_parameters=1)
        (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect)
        (5): Dropout(p=False, inplace=False)
        (6): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      )
    )
  )
)

Before we go over the training loop, we have to import and initialize a training module. The training module sets up the training, validation, and inference steps of the model, as well as the optimizer. During the training and validation steps it also calls the loss function `loss_fn` that is used to monitor the training process.

In [3]:
from resource_aware_ml.training.trainer import TrainModule

from torch.optim import AdamW
from torch.nn import L1Loss

In [4]:
# Change the Optimizer or loss function here.
train_module = TrainModule(model=model, loss_fn=L1Loss(), optimizer=AdamW, lr=1e-3)

We will also need a data module that handles the data loading for us. The data module implemented in the package is designed to load the dataset that is provided in the repository. The data module inherits from the [↗`lightning.LightningDataModule`](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.core.LightningDataModule.html) and is simply a collection of dataloaders for the training, validation, test, and inference stages.

In [5]:
from resource_aware_ml.io.data import H5DataModule

In [6]:
data_module = H5DataModule(
    data_dir="./data",  # Path to the data in the repository relative to this notebook
    batch_size=20,      # Change if you run out of memory
    fourier=True,       # Our data is in Fourier space
    num_workers=4       # Change the number of concurrent CPU cores loading the data
)

## Task 1: Logging

Implement a logger that logs the experiment data such as the train loss and validation loss to **MLFlow**. Have a look at the lightning docs for more information on [loggers](https://lightning.ai/docs/pytorch/stable/api_references.html#loggers). Save the logger in a variable `logger`.

In [7]:
# Implement your solution here
from pathlib import Path

from lightning.pytorch.loggers import CSVLogger, MLFlowLogger


mlruns_dir = Path("./build/mlruns").expanduser().resolve()
logger = [CSVLogger(save_dir="./build"), MLFlowLogger(save_dir=mlruns_dir)]

/home/anno/.local/conda/envs/resource-awareness/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Now we can set up the [↗`lightning.Trainer`](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.trainer.trainer.Trainer.html). Here, we can set

In [8]:
from lightning import Trainer
from lightning.pytorch.callbacks import RichProgressBar

In [9]:
trainer = Trainer(
    max_epochs=10,
    accelerator="auto",  # Change to gpu, or cpu, if you like
    precision="32-true",
    logger=logger,
    callbacks=RichProgressBar(),
    log_every_n_steps=20,  # set to batch size
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


## Task 2: Emission Tracking

To start the training process, we will have to use the `Trainer.fit` method. In order to track the emissions, we will need to use CodeCarbon or the Lamarr Energy Tracker (LET).

In [10]:
import pandas as pd
from lamarr_energy_tracker import EnergyTracker

Your task is to log the following metrics and parameters using MLFlow and CodeCarbon/LET:
<table>
<tr><td>
    
| Metric                         | Variable Name              |
| ------------------------------ | -------------------------- |
| Number of trainable parameters | `num_trainable_parameters` |
| Total runtime                  | `running_time_total`       |
| Runtime per epoch              | `running_time`             |
| Total power draw               | `power_draw_total`         |
| Power draw per epoch           | `power_draw`               |

</td><td>

| Parameter                        | Variable Name  |
| -------------------------------- | -------------- |
| Model name                       | `model`        |
| Dataset                          | `dataset`      |
| Architecture (GPU Model)         | `architecture` |
| Task (`training` or `inference`) | `task`         |

</td></tr> </table>

In [20]:
emissions_path = Path("./build").expanduser().resolve()

# Track the training loop
with EnergyTracker(project_name="dpg_tutorial", output_dir=emissions_path):
    trainer.fit(model=train_module, datamodule=data_module)

# When using multiple loggers, get the experiment from the MLFLowLogger instance
mlflow_logger = next(
    logger
    for logger in trainer.loggers
    if isinstance(logger, MLFlowLogger)
)
experiment = mlflow_logger.experiment
run_id = mlflow_logger._run_id

# Get total number of samples (train + valid)
num_samples = trainer.datamodule.train_length + trainer.datamodule.valid_length

emissions_path /= "emissions.csv"
emission_data = pd.read_csv(emissions_path).to_dict()

kwh = 3.6e6
metrics = dict(
    running_time_total=emission_data["duration"][0],
    running_time=emission_data["duration"][0] / num_samples,
    power_draw_total=emission_data["energy_consumed"][0] * kwh,
    power_draw=emission_data["energy_consumed"][0] * kwh / num_samples,
)

for key, val in metrics.items():
    experiment.log_metric(
        key=key,
        value=val,
        run_id=run_id
    )

model_name = trainer.model.model.__class__.__name__
model_name += "_" + trainer.model.optimizer.__name__
model_name += "_" + trainer.model.loss_fn.__class__.__name__

params = dict(
    model=model_name,
    dataset="Radio Simulation HDF5",
    architecture=emission_data["gpu_model"][0],
    task="train"
)

/home/anno/.local/conda/envs/resource-awareness/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory ./build/lightning_logs/version_3/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ SRResNet6 │  157 K │ train │     0 │
│ 1 │ loss_fn │ L1Loss    │      0 │ train │     0 │
└───┴─────────┴───────────┴────────┴───────┴───────┘

Trainable params: 157 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 157 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 31                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/anno/.local/conda/envs/resource-awareness/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py
:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

`Trainer.fit` stopped: `max_epochs=10` reached.



Tracker stopped - this experiment consumed 0.018 Wh.


Using CodeCarbon 3.0.8, the energy consumption of running all experiments on an Intel(R) Core(TM) i7-8700K CPU and 1 x NVIDIA GeForce RTX 2080 is estimated to 3.132 Wh.This corresponds to estimated carbon emissions of 1.190 gCO2-equivalents, assuming a carbon intensity of 380 gCO2/kWh~\cite{lamarr_energy_tracker,codecarbon}. Note that these numbers are underestimations of actual resource consumption and do not account for overhead factors or embodied impacts~\cite{ai_energy_validation}.



## Task 3: MLFlow Export

Export the logged data to a `.csv` file using MLFlow's CLI tool.

## Task 4: Visualisation Using STREP

Visualise the data using the STREP framework.